# Exercise 5.8 — Linear Regression Lab
**Course:** 23CSE301 Machine Learning
**Section:** 5.8 Exercises (Lab Activity Book)

This notebook completes both exercises:
1. Predict the fare of a taxi ride in Chicago, Illinois (City of Chicago Taxi Trips dataset).
2. Conduct Linear Regression to predict body weight from height, age, and gender.


## Part 1: Chicago Taxi Fare Prediction

**Dataset:** [City of Chicago — Taxi Trips](https://data.cityofchicago.org/Transportation/Taxi-Trips/wrvz-psew)
(Socrata Open Data / SODA API, dataset id `wrvz-psew`)

The full dataset has 100M+ rows, so instead of downloading the whole CSV we pull a
manageable sample directly from the Socrata API using a SoQL query. This works
directly in Google Colab (internet access required).


In [ ]:
# Install/upgrade requests if needed (Colab already has it)
!pip install -q pandas scikit-learn matplotlib requests

In [ ]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# ---- Pull a sample of the Chicago Taxi Trips dataset via the Socrata API ----
# Dataset landing page: https://data.cityofchicago.org/Transportation/Taxi-Trips/wrvz-psew
# API endpoint (JSON):  https://data.cityofchicago.org/resource/wrvz-psew.json

BASE_URL = "https://data.cityofchicago.org/resource/wrvz-psew.json"

# SoQL query: pull only the columns we need, filter out null/zero fares & distances,
# and limit to a workable sample size (50,000 rows).
params = {
    "$select": "trip_seconds,trip_miles,fare,trip_start_timestamp",
    "$where": "fare > 0 AND trip_miles > 0 AND trip_seconds > 0",
    "$limit": 50000
}

response = requests.get(BASE_URL, params=params)
response.raise_for_status()
data = pd.DataFrame(response.json())
print(data.shape)
data.head()

In [ ]:
# ---- If you don't have internet access (or want a fixed offline file), ----
# ---- uncomment the two lines below to load a previously-saved CSV instead ----
# data = pd.read_csv('/content/chicago_taxi_sample.csv')
# print(data.shape)

In [ ]:
# Convert numeric columns (the API returns everything as strings)
data['trip_seconds'] = pd.to_numeric(data['trip_seconds'], errors='coerce')
data['trip_miles']   = pd.to_numeric(data['trip_miles'], errors='coerce')
data['fare']         = pd.to_numeric(data['fare'], errors='coerce')

data.isnull().sum()

In [ ]:
# Drop any rows with missing/invalid values
data = data.dropna(subset=['trip_seconds', 'trip_miles', 'fare'])

# Remove extreme outliers (e.g. unrealistic fares or trip lengths)
data = data[(data['fare'] < 200) & (data['trip_miles'] < 100) & (data['trip_seconds'] < 3*3600)]
print(data.shape)
data.describe()

In [ ]:
# ---- Assigning dependent and independent variables ----
x = data[['trip_miles', 'trip_seconds']].values
y = data['fare'].values

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
print(x_train.shape, x_test.shape)

In [ ]:
# ---- Fitting the Model (Linear Regression) ----
model = LinearRegression()
model.fit(x_train, y_train)
y_pred = model.predict(x_test)

print('Coefficients (trip_miles, trip_seconds):', model.coef_)
print('Intercept:', model.intercept_)

In [ ]:
# ---- Model Evaluation ----
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print('MSE  :', mse)
print('RMSE :', rmse)
print('R2   :', r2)

In [ ]:
# ---- Plot: Actual vs Predicted Fare ----
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred, alpha=0.3, color='blue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linewidth=2)
plt.title('Chicago Taxi Fare: Actual vs Predicted')
plt.xlabel('Actual Fare ($)')
plt.ylabel('Predicted Fare ($)')
plt.show()

In [ ]:
# ---- Plot: Fare vs Trip Miles (single-feature view) ----
plt.figure(figsize=(7,5))
plt.scatter(data['trip_miles'], data['fare'], s=5, alpha=0.3, color='green')
plt.title('Fare vs Trip Distance')
plt.xlabel('Trip Miles')
plt.ylabel('Fare ($)')
plt.show()

---
## Part 2: Predicting Body Weight (Height, Age, Gender)

**Dataset:** the 10-row sample table from the Lab Activity Book (Section 5.8, Exercise 2).
A ready-to-use CSV (`weight_dataset.csv`) is provided alongside this notebook — in Colab,
upload it via the Files panel, or recreate it with the code below.


In [ ]:
# ---- Recreate the dataset (or use the uploaded weight_dataset.csv) ----
weight_data = {
    'Weight': [79, 69, 73, 95, 82, 55, 69, 71, 64, 69],
    'Height': [1.80, 1.68, 1.82, 1.70, 1.87, 1.55, 1.50, 1.78, 1.67, 1.64],
    'Age':    [35, 39, 25, 60, 27, 18, 89, 42, 16, 52],
    'Gender': ['Male','Male','Male','Male','Male','Female','Female','Female','Female','Female']
}
data2 = pd.DataFrame(weight_data)

# If you uploaded the CSV in Colab instead, use:
# data2 = pd.read_csv('/content/weight_dataset.csv')

data2

In [ ]:
# ---- Checking for any null values ----
data2.isnull().sum()

In [ ]:
# ---- Encode Gender (categorical -> numeric) ----
data2['Gender'] = data2['Gender'].map({'Male': 1, 'Female': 0})
data2

In [ ]:
# ---- Assigning dependent and independent variables ----
x2 = data2[['Height', 'Age', 'Gender']].values
y2 = data2['Weight'].values

# NOTE: this dataset only has 10 rows, so the test split is tiny by design —
# this exercise is meant to illustrate the workflow, not to build a production model.
x2_train, x2_test, y2_train, y2_test = train_test_split(x2, y2, test_size=0.2, random_state=42)
print(x2_train.shape, x2_test.shape)

In [ ]:
# ---- Fitting the Model (Linear Regression) ----
model2 = LinearRegression()
model2.fit(x2_train, y2_train)
y2_pred = model2.predict(x2_test)

print('Predicted:', y2_pred)
print('Actual   :', y2_test)

In [ ]:
# ---- Model Evaluation ----
mse2 = mean_squared_error(y2_test, y2_pred)
rmse2 = np.sqrt(mse2)
r2_2 = r2_score(y2_test, y2_pred)

print('MSE  :', mse2)
print('RMSE :', rmse2)
print('R2   :', r2_2)

print('\nCoefficients [Height, Age, Gender]:', model2.coef_)
print('Intercept:', model2.intercept_)

In [ ]:
# ---- Plot: Weight vs Height, coloured by Gender ----
plt.figure(figsize=(7,5))
colors = data2['Gender'].map({1: 'blue', 0: 'magenta'})
plt.scatter(data2['Height'], data2['Weight'], c=colors, s=80)
plt.title('Weight vs Height (blue = Male, magenta = Female)')
plt.xlabel('Height (m)')
plt.ylabel('Weight (kg)')
plt.show()

In [ ]:
# ---- Predict weight for a new person: Height=1.75m, Age=30, Gender=Male ----
new_person = np.array([[1.75, 30, 1]])
predicted_weight = model2.predict(new_person)
print(f'Predicted weight: {predicted_weight[0]:.2f} kg')

---
### Notes
- **Part 1** uses the live Socrata API for the City of Chicago Taxi Trips dataset, so it
  needs an internet connection when run (works out-of-the-box in Google Colab). If the API
  is rate-limited or unavailable, save a pulled sample once as `chicago_taxi_sample.csv`
  and reload it from disk using the commented-out line.
- **Part 2** uses the small 10-row example table from the lab book directly. Because the
  dataset is so small, treat the train/test split and metrics as illustrative of the
  *workflow* (assign variables → split → fit → evaluate) rather than a statistically
  robust model.
- Submit this Colab/GitHub link or the `.ipynb` file as instructed in the lab book.
